In [ ]:
import requests

def check_url_status(url, timeout=5):
    """
    Checks the HTTP status of a given URL using a HEAD request.
    Returns True if the status code is not 404, False otherwise (including connection errors).
    """
    if not url: # Handle empty URLs directly
        return False
    try:
        response = requests.head(url, timeout=timeout, allow_redirects=True)
        return response.status_code != 404 # Modified to filter only 404
    except requests.exceptions.RequestException as e:
        # Catch all request-related exceptions (connection errors, timeouts, etc.)
        # print(f"Error checking URL {url}: {e}") # Uncomment for debugging
        return False

In [ ]:
! pip install datatrove

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-1.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.9.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=9b21ab1540c995c2819aabd739dc491de7f914c409202c025a7455c00a5b7d4c
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


# Data Collection from Multiple Hugging Face Datasets

This section collects 30 random samples from three additional Hugging Face datasets: `finepdfs`, `c4`, and `RedPajama-V2 sample`. Each dataset's samples will be saved to its own JSON file.

# Unified Data Collection Pipeline

This section defines a single pipeline to collect 30 random samples from each of the specified Hugging Face datasets: `fineweb`, `finepdfs`, `c4`, and `RedPajama-V2 sample`. Each dataset's samples will be saved to its own JSON file.

In [ ]:
from datasets import load_dataset

In [ ]:
fw = load_dataset("HuggingFaceFW/finepdfs", name="eng_Latn", split="train", streaming=True)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

In [ ]:
for doc in fw:
  print(doc)
  break

{'text': 'SSMU LEGISLATIVE COUNCIL AGENDA\n\nFEBRUARY 23, 2017\n\n1. Call to Order\n2. Land Acknowledgement\n3. Attendance\n4. Approval of the Minutes\n5. Guest Speakers\na. McGill Peer Support Centre\n6. Adoption of the Agenda\n7. Question Period (5)\n8. Report of the Steering Committee (2)\n9. Announcements (5)\n10. Old Business\na. Motion Regarding the Amendment of Internal Regulations of Governance\nb. Motion Regarding the Endorsement of the McGill Communities Council Letter to the Board of Governors\nc. Motion Regarding Athletics Ancillary Fee Referendum Question\nd. Motion Regarding Referendum Question on Constitutional Amendments\ne. Motion Regarding Adoption of a Policy Against Unpaid Internships\nf. Motion Regarding Revisions to the Equity Policy\ng. Motion Regarding Revisions to the Indigenous Solidarity Policy\n11. New Business\na. Motion Regarding SSMU Support For Floor Fellow Bargaining\nb. Motion Regarding Condemnation of Transphobic Event at the Newman Centre\n\n12. Repo

In [ ]:
from langdetect import detect, DetectorFactory

# Ensure reproducible results by seeding the detector factory
DetectorFactory.seed = 0

import json
import random
from datatrove.pipeline.readers import ParquetReader
from datasets import load_dataset # Added load_dataset

# Define all datasets to process along with their Hugging Face paths and respective reader types
# RedPajama-V2 and C4 will be handled separately using datasets.load_dataset due to format/script issues or user preference.
all_datasets_info = [
    ("fineweb", "hf://datasets/HuggingFaceFW/fineweb/data", ParquetReader)
    # finepdfs will now be handled separately using datasets.load_dataset
]

num_samples_per_dataset = 50
# Read a larger number of documents to allow for better random sampling from the stream
intermediate_read_limit = 200

all_processed_data = {}

# --- Datatrove pipeline processing ---
for dataset_name, dataset_path, ReaderType in all_datasets_info:
    print(f"\n{'='*50}\nProcessing dataset: {dataset_name} from {dataset_path} using Datatrove")

    collected_documents_for_sampling = []
    try:
        reader = ReaderType(dataset_path, limit=intermediate_read_limit)
        for doc in reader():
            url_in_metadata = doc.metadata.get("url")
            url_exists = bool(url_in_metadata)
            content_exists = bool(doc.text and doc.text.strip())
            url_is_valid = False
            lang_is_en = False

            if url_exists:
                url_is_valid = check_url_status(url_in_metadata)

            if content_exists and len(doc.text) > 50: # Minimum length for reliable language detection
                try:
                    if detect(doc.text) == 'en':
                        lang_is_en = True
                except Exception:
                    # langdetect can fail on very short or ambiguous strings
                    print(f"Skipping document from {dataset_name} due to language detection error on text.")

            if url_exists and content_exists and url_is_valid and lang_is_en:
                collected_documents_for_sampling.append(doc)
            else:
                if not url_exists:
                    print(f"Skipping document from {dataset_name} due to missing or empty URL in metadata.")
                elif not content_exists:
                    print(f"Skipping document from {dataset_name} due to missing or empty text content.")
                elif not url_is_valid:
                    print(f"Skipping document from {dataset_name} due to invalid URL status for URL: {url_in_metadata}")
                elif not lang_is_en:
                    print(f"Skipping document from {dataset_name} due to non-English content detected or language detection failure.")

    except Exception as e:
        print(f"Error reading documents from {dataset_name}: {e}")
        continue

    if len(collected_documents_for_sampling) >= num_samples_per_dataset:
        selected_documents = random.sample(collected_documents_for_sampling, num_samples_per_dataset)
    else:
        selected_documents = collected_documents_for_sampling
        print(f"Warning: Only {len(selected_documents)} documents available for {dataset_name}. Expected {num_samples_per_dataset}.")

    if not selected_documents:
        print(f"No documents found or selected for {dataset_name} at {dataset_path}. Skipping save.")
        continue

    processed_documents = []
    for doc in selected_documents:
        processed_documents.append({
            "id": doc.id,
            "text": doc.text,
            "media": doc.media,
            "metadata": doc.metadata
        })

    all_processed_data[dataset_name] = processed_documents
    output_filename = f"{dataset_name}_documents.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(processed_documents, f, ensure_ascii=False, indent=4)

    print(f"Successfully saved {len(processed_documents)} random documents from {dataset_name} to {output_filename}")

# --- finepdfs processing using datasets.load_dataset ---
finepdfs_dataset_name = "finepdfs"
print(f"\n{'='*50}\nProcessing dataset: {finepdfs_dataset_name} using datasets.load_dataset")

try:
    # Load finepdfs directly using datasets.load_dataset with the specified language
    finepdfs_stream = load_dataset("HuggingFaceFW/finepdfs", name="eng_Latn", split="train", streaming=True)

    sampled_finepdfs_docs = []
    for i, doc in enumerate(finepdfs_stream):
        doc_url = doc.get("url")
        url_exists = bool(doc_url)
        content_exists = bool(doc.get("text") and doc.get("text").strip())
        url_is_valid = False
        lang_is_en = False # The dataset is already filtered by eng_Latn, but we re-check for consistency

        if url_exists:
            url_is_valid = check_url_status(doc_url)

        if content_exists and len(doc.get("text")) > 50: # Minimum length for reliable language detection
            try:
                if detect(doc.get("text")) == 'en':
                    lang_is_en = True
            except Exception:
                print(f"Skipping document from {finepdfs_dataset_name} due to language detection error on text.")

        if url_exists and content_exists and url_is_valid and lang_is_en:
            sampled_finepdfs_docs.append(doc)
        else:
            if not url_exists:
                print(f"Skipping document from {finepdfs_dataset_name} due to missing or empty URL.")
            elif not content_exists:
                print(f"Skipping document from {finepdfs_dataset_name} due to missing or empty text content.")
            elif not url_is_valid:
                print(f"Skipping document from {finepdfs_dataset_name} due to invalid URL status for URL: {doc_url}")
            elif not lang_is_en:
                print(f"Skipping document from {finepdfs_dataset_name} due to non-English content detected or language detection failure (despite `name='eng_Latn'` filter).")

        if len(sampled_finepdfs_docs) >= num_samples_per_dataset:
            break

    processed_finepdfs_docs = []
    if sampled_finepdfs_docs: # Only sample if there are enough documents
        if len(sampled_finepdfs_docs) >= num_samples_per_dataset:
            selected_finepdfs_docs = random.sample(sampled_finepdfs_docs, num_samples_per_dataset)
        else:
            selected_finepdfs_docs = sampled_finepdfs_docs
            print(f"Warning: Only {len(selected_finepdfs_docs)} documents available for {finepdfs_dataset_name}. Expected {num_samples_per_dataset}.")
    else:
        selected_finepdfs_docs = []
        print(f"No documents found or selected for {finepdfs_dataset_name}. Skipping save.")


    if selected_finepdfs_docs:
        for doc in selected_finepdfs_docs:
            processed_finepdfs_docs.append({
                "text": doc.get("text"),
                "metadata": {
                    "url": doc.get("url"),
                    "id": doc.get("id"),
                    "date": doc.get("date"),
                    "language": doc.get("language")
                }
            })

        all_processed_data[finepdfs_dataset_name] = processed_finepdfs_docs
        output_filename = f"{finepdfs_dataset_name}_documents.json"
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(processed_finepdfs_docs, f, ensure_ascii=False, indent=4)

        print(f"Successfully saved {len(processed_finepdfs_docs)} random documents from {finepdfs_dataset_name} to {output_filename}")
    else:
        print(f"No documents processed for {finepdfs_dataset_name}.")


except Exception as e:
    print(f"Error loading or processing {finepdfs_dataset_name} dataset: {e}")


# --- Summary printing (kept as is) ---
print(f"\n{'='*50}\nSummary of first few documents from each dataset processed:") # Changed text for clarity
for dataset_name, docs in all_processed_data.items():
    if docs:
        print(f"\n--- {dataset_name} (first {min(3, len(docs))} documents) ---")
        display(docs[:min(3, len(docs))])
    else:
        print(f"\n--- No documents processed for {dataset_name} ---")



Processing dataset: fineweb from hf://datasets/HuggingFaceFW/fineweb/data using Datatrove


2026-06-09 13:56:22.528 | INFO     | datatrove.pipeline.readers.base:read_files_shard:206 - Reading input file CC-MAIN-2013-20/000_00000.parquet, 1/27468


Skipping document from fineweb due to invalid URL status for URL: http://1075zoofm.com/listeners-get-sky-high-view-of-missoula-from-hot-air-balloons/
Skipping document from fineweb due to invalid URL status for URL: http://12vspotlight.com/
Skipping document from fineweb due to invalid URL status for URL: http://2012indyinfo.com/category/sfhs/
Skipping document from fineweb due to invalid URL status for URL: http://49ersnews.com/2008/09/spencer-done/
Skipping document from fineweb due to invalid URL status for URL: http://4mothers1blog.com/2012/04/24/a-big-silly-distraction/
Skipping document from fineweb due to invalid URL status for URL: http://6shooter.co.nz/
Skipping document from fineweb due to invalid URL status for URL: http://a.blip.tv/statesidefootytv/stateside-footy-episode-12-16-usafl-womens-final-and-2012-in-review-6540732
Skipping document from fineweb due to invalid URL status for URL: http://aacap.org/page.ww?name=When+to+Seek+Help+for+Your+Child&section=Facts+for+Famili

Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

Skipping document from finepdfs due to invalid URL status for URL: https://www.lac.org.na/laws/1975/og3446.pdf
Skipping document from finepdfs due to invalid URL status for URL: https://eprocurement.uaeu.ac.ae/docs/complaint_procedure.pdf
Skipping document from finepdfs due to non-English content detected or language detection failure (despite `name='eng_Latn'` filter).
Skipping document from finepdfs due to non-English content detected or language detection failure (despite `name='eng_Latn'` filter).
Skipping document from finepdfs due to invalid URL status for URL: https://www.treasury.gov/resource-center/fin-mkts/Documents/TRIP_Form_02B_Schedule_C.pdf
Skipping document from finepdfs due to invalid URL status for URL: http://www.hkex.com.hk/eng/market/sec_tradinfra/chinaconnect/Documents/List%20of%20Eligible%20Stocks%20for%20Northbound%20Trading%20(Eng).pdf
Skipping document from finepdfs due to non-English content detected or language detection failure (despite `name='eng_Latn'` fil

[{'id': '<urn:uuid:0868921d-8323-4a3d-b012-51c15c046cc1>',
  'text': "When I found out we would be getting a PopATot for review I was excited! I knew before we even had it that we would like it, but I had no idea how much we would LOVE it! I am so excited to share this amazing product with you all!\nPopATot was created by parents, Evan and Stacy, who have 5 children! I personally think this idea is wonderful. When I was trying to describe to my husband what was coming in the mail for Ladybug, I described it as a cross between a card table, an exersaucer, and a fabric folding chair. He sort of got it, but when he finally saw it he totally understood my references!!!\nI will let our many photos of Ladybug speak mostly for themselves (she is 5 1/2-6 months in all of these photos). We use this thing every day now, and in many different places throughout each day-it is that easy to move around.\nMy absolute favorite use for the PopATot is in our schoolroom. We have a fairly small schoolroom


--- finepdfs (first 3 documents) ---


[{'text': "ACAA  LEAGUE RECORD\n\nMEADOWBROOK CHRISTIAN SCHOOL\n\nSENIOR HIGH TRACK AND FIELD RECORDS (Program began in 2000)\n\n| EVENT | NAME (YEAR) | TIME / DISTANCE | NAME (YEAR) | TIME/ DISTANCE |\n|---|---|---|---|---|\n| 100 METER | Karolyn Huber – 2004 | :13.19 | Stephen Moore - 2006 | :11.53 |\n| 200 METER | Karolyn Huber - 2004 | :27.97 | Jose Rivas- 2014 | :23.94 |\n| 400 METER | Melissa Balliet - 2006 | 1:05.10 | Andy Willits- 2009 | :52.50  |\n| 800 METER | Stefanie Rowe – 2004 Adrienne Yordy- 2013 | 2:27.18 2:40.82 | J.J. Brooks – 2005 J.J. Brooks – 2004 | 2:05.87 2:09.74  |\n| 1600 METER | Stefanie Rowe – 2004 Stefanie Rowe – 2004 | 5:29.30 5:33.88  | J.J. Brooks – 2005 J.J. Brooks - 2004 | 4:35.19 4:54.38  |\n| 3200 METER | Stefanie Rowe – 2004 | 12:46.00 | Kurtis Klodnicki - 2012 | 11:00.88 |\n| 4X100 RELAY | Madison Burrows, Kennedy Barner, Lizzie Yordy, Paige Friesema - 2012 | :53.76  | A.Troxell, E. Hartle, T. Morton, S. Moore – 2006 Matthew Sweigard, John Bossert, 

# Loading RedPajama-Data-V2 Sample using `datasets` library

Since the `datatrove` readers and the default `load_dataset` script for `RedPajama-Data-V2` faced issues, this section demonstrates how to load the `sample` split directly using `datasets.load_dataset` by specifying the `data_files` argument. This allows for direct loading of the `.jsonl.gz` files from the Hugging Face Hub.

In [ ]:
redpajama_data_files = {
    'train': '/resolve/main/sample/documents/2023-06/*/'
}

In [ ]:
import requests
import gzip
import json
from io import BytesIO
import random # Added import

base = (
    "https://huggingface.co/datasets/"
    "togethercomputer/RedPajama-Data-V2/resolve/main/"
    "sample/documents/2023-06"
)

all_collected_docs = [] # List to collect all documents read (up to 3 per file)

for shard_idx in range(10):
    urls = [
        f"{base}/{shard_idx:04d}/en_head.json.gz",
        f"{base}/{shard_idx:04d}/en_middle.json.gz",
    ]
    for url in urls:
      try:
          r = requests.get(url, timeout=30)
          r.raise_for_status()

          with gzip.GzipFile(fileobj=BytesIO(r.content)) as f:
              # Read up to the first 3 documents from each gzipped file
              for i in range(15):
                  try:
                      line = next(f)
                      doc = json.loads(line)

                      # Check if site exists (by checking for a URL) and there is some content (text)
                      doc_url = doc.get("url")
                      url_exists = bool(doc_url)
                      content_exists = bool(doc.get("raw_content") and doc.get("raw_content").strip()) # RedPajama uses 'raw_content'
                      url_is_200 = False

                      if url_exists:
                          url_is_200 = check_url_status(doc_url)

                      if url_exists and content_exists and url_is_200:
                          all_collected_docs.append(doc)
                      else:
                          if not url_exists:
                              print(f"Skipping document from RedPajama-Data-V2 due to missing or empty URL in metadata for shard {shard_idx:04d}.")
                          elif not content_exists:
                              print(f"Skipping document from RedPajama-Data-V2 due to missing or empty text content for shard {shard_idx:04d}.")
                          elif not url_is_200:
                              print(f"Skipping document from RedPajama-Data-V2 due to non-200 status for URL: {doc_url} (shard {shard_idx:04d}).")
                  except StopIteration:
                      break # No more lines in this file

          print(f"Loaded up to 3 documents from shard {shard_idx:04d} part of {url.split('/')[-1]}")

      except Exception as e:
          print(f"Failed to load from {url} for shard {shard_idx:04d}: {e}")

# Now, sample 30 random documents from all collected documents that passed the filters
num_samples_to_take = 100
if len(all_collected_docs) >= num_samples_to_take:
    shards = random.sample(all_collected_docs, num_samples_to_take)
else:
    shards = all_collected_docs
    print(f"Warning: Only {len(shards)} documents collected in total. Expected {num_samples_to_take} for sampling.")

try:
    # Convert to a list of dictionaries for consistent output format
    processed_redpajama_docs = []
    for doc in shards:
        # The structure of documents from datasets.load_dataset might differ slightly
        # from datatrove.Document, so we extract relevant fields.
        processed_redpajama_docs.append({
            "text": doc.get("raw_content"), # Assuming 'text' field exists
            "metadata": {
                "url": doc.get("url"),
                "source": doc.get("source"),
                "redpajama_id": doc.get("digest") # Use a different key to avoid conflicts if 'id' from datatrove was unique
            }
        })

    # Save to a JSON file
    output_filename = "redpajama_v2_sample_documents.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(processed_redpajama_docs, f, ensure_ascii=False, indent=4)

    print(f"Successfully saved {len(processed_redpajama_docs)} random documents from RedPajama-Data-V2 sample to {output_filename}")

    print(f"\n{'='*50}\nExample of documents from RedPajama-Data-V2/sample (first {min(3, len(processed_redpajama_docs))} documents):")
    display(processed_redpajama_docs[:min(3, len(processed_redpajama_docs))])

except Exception as e:
    print(f"Error loading or processing RedPajama-Data-V2/sample: {e}")

Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://archives.library.rice.edu/repositories/2/archival_objects/79290 (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://ateamtriad.com/testimonials/ (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://bastum.us/2022/11/05/ (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://billstatus.ls.state.ms.us/documents/2007/html/SC/SC0542IN.htm (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://bluegrass-conceptsgifts.com/all-you-need-to-learn-about-biography/ (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://bluegrass-conceptsgifts.com/tips-on-composing-a-bio/ (shard 0000).
Skipping document from RedPajama-Data-V2 due to non-200 status for URL: http://butthenwhat.com/2011/04/14/obamas-trigger-option/ (shard 0000).
Lo

[{'text': 'I have a couple of signs hanging in the Mustang cave. The Goodyear sign is my favorite and actually a bit of a mystery. Here\'s the info I have:\nIt\'s 100 inches long and made of solid brass with blue enamel lettering. I bought it in 2005 at an auction here in Denmark. I asked the seller where it came from and he told me he found it in 1988 while digging in his garden. He was curious to know how a thing like that could end up in his garden. It turned out that until the 1940\'s the area was used as a junkyard. It\'s still strange though how a sign like that could end up in Denmark. Some think it\'s a sign from a merchant ship and that the ship ended its days here but I doubt that.\nGoodyear Tire & Rubber Company was founded in 1898, so maybe the sign is from a time before then. Perhaps from Charles Goodyear\'s earlier years (google his name and check his history - tough life). When I google the exact words "Goodyear Rubber Co. New York", not much more than an add from 1884 i

# Loading C4 (Colossal, Cleaned Common Crawl) using `datasets` library

This section loads the C4 dataset, specifically the 'en' configuration, using `datasets.load_dataset` with `streaming=True`. Due to the nature of streaming datasets, we will collect the first 30 documents from the stream and save them to a JSON file. True random sampling of the entire C4 dataset in streaming mode is not directly supported.

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

from datasets import load_dataset
import json

num_samples_to_take = 30

print("\n==================================================\nLoading C4 'en' dataset in streaming mode...")

try:
    c4_en_dataset_stream = load_dataset("allenai/c4", "en", streaming=True, split='train')

    sampled_c4_docs = []
    for i, doc in enumerate(c4_en_dataset_stream):
        doc_url = doc.get("url")
        url_exists = bool(doc_url)
        content_exists = bool(doc.get("text") and doc.get("text").strip())
        url_is_valid = False
        lang_is_en = False

        if url_exists:
            url_is_valid = check_url_status(doc_url)

        if content_exists and len(doc.get("text")) > 50:
            try:
                if detect(doc.get("text")) == 'en':
                    lang_is_en = True
            except Exception:
                print(f"Skipping document from C4 'en' due to language detection error on text.")

        if url_exists and content_exists and url_is_valid and lang_is_en:
            sampled_c4_docs.append(doc)
        else:
            if not url_exists:
                print(f"Skipping document from C4 'en' due to missing or empty URL.")
            elif not content_exists:
                print(f"Skipping document from C4 'en' due to missing or empty text content.")
            elif not url_is_valid:
                print(f"Skipping document from C4 'en' due to invalid URL status for URL: {doc_url}")
            elif not lang_is_en:
                print(f"Skipping document from C4 'en' due to non-English content detected or language detection failure.")

        if len(sampled_c4_docs) >= num_samples_to_take:
            break

    processed_c4_docs = []
    for doc in sampled_c4_docs:
        print(doc)
        processed_c4_docs.append({
            "text": doc.get("text"),
            "metadata": {
                "url": doc.get("url"),
                "timestamp": doc.get("timestamp"),
                "c4_id": doc.get("id")
            }
        })

    output_filename = "c4_en_documents.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(processed_c4_docs, f, ensure_ascii=False, indent=4)

    print(f"Successfully saved {len(processed_c4_docs)} documents from C4 'en' to {output_filename}")

    print(f"\n{'='*50}\nExample of documents from C4 'en' (first {min(3, len(processed_c4_docs))} documents):")
    display(processed_c4_docs[:min(3, len(processed_c4_docs))])

except Exception as e:
    print(f"Error loading or processing C4 'en' dataset: {e}")


Loading C4 'en' dataset in streaming mode...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Skipping document from C4 'en' due to invalid URL status for URL: https://klyq.com/beginners-bbq-class-taking-place-in-missoula/
Skipping document from C4 'en' due to invalid URL status for URL: https://forums.macrumors.com/threads/restore-from-larger-disk-to-smaller-disk.1311329/
Skipping document from C4 'en' due to invalid URL status for URL: https://awishcometrue.com/Catalogs/Clearance/Tweens/V1960-Find-A-Way
Skipping document from C4 'en' due to invalid URL status for URL: https://www.blackhatworld.com/seo/how-many-backlinks-per-day-for-new-site.258615/
Skipping document from C4 'en' due to invalid URL status for URL: http://bond.dpsk12.org/category/news/
Skipping document from C4 'en' due to invalid URL status for URL: https://tatkalforsure.com/trains-between-stations/bangalore-cy-junction-sbc-to-gondia-junction-g/
Skipping document from C4 'en' due to invalid URL status for URL: http://www.iammeek.com/2018/06/the-rich-get-richer-and-poor-get-poorer.html
Skipping document from C4

[{'text': 'I thought I was going to finish the 3rd season of the Wire tonight.\nBut there was a commentary on episode 11, so I had to re-watch Middle Ground with the commentary. Hopefully I can finish the season next weekend.',
  'metadata': {'url': 'https://karaokegal.livejournal.com/1773485.html',
   'timestamp': '2019-04-18 14:16:05',
   'c4_id': None}},
 {'text': 'Biomedics 1 Day Extra are daily replacement disposable contact lenses by CooperVision Hydron. Buy one box of 90 lenses.\nBiomedics 1 Day Extra contacts give you all the convenience of a daily disposable lens with no need for solutions, cases or cleaning and are perfect for the occasional wear. These lenses have greater comfort handling with superior ease of insertion and removal.\nBiomedic 1 Day Extra are also marketed under various other brand names including Clear Choice 1-day, Ascend 1-day, easyvision CLARISION SPHERE, Clearsight 1 Day and ProView Daily Disposable.',
  'metadata': {'url': 'https://www.webcontacts.com.a

### Organize JSON files into a dedicated folder

To keep our workspace tidy, let's move all the collected document JSON files into a new folder.

In [ ]:
import os
import shutil

# Define the folder name for collected JSON files
output_folder = "collected_json_data"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)
print(f"Created folder: {output_folder}")

# List of JSON files to move
json_files_to_move = [
    "fineweb_documents.json",
    "finepdfs_documents.json",
    "redpajama_v2_sample_documents.json",
    "c4_en_documents.json"
]

# Move each file into the new folder
for filename in json_files_to_move:
    if os.path.exists(filename):
        shutil.move(filename, os.path.join(output_folder, filename))
        print(f"Moved '{filename}' to '{output_folder}/'")
    else:
        print(f"File '{filename}' not found, skipping.")

print("All specified JSON files have been organized.")

Created folder: collected_json_data
Moved 'fineweb_documents.json' to 'collected_json_data/'
Moved 'finepdfs_documents.json' to 'collected_json_data/'
Moved 'redpajama_v2_sample_documents.json' to 'collected_json_data/'
Moved 'c4_en_documents.json' to 'collected_json_data/'
All specified JSON files have been organized.


### Merge all JSON files into a single dataset

Now, let's combine all the individual JSON files collected from different datasets into one master JSON file. This will make it easier to work with the data as a single unit.

In [ ]:
import os
import json

# Define the folder where the JSON files are located
output_folder = "collected_json_data"

# List to hold all documents from all datasets
combined_dataset = []

# Iterate over all files in the output_folder
for filename in os.listdir(output_folder):
    if filename.endswith(".json"): # Ensure we only process JSON files
        filepath = os.path.join(output_folder, filename)
        print(f"Reading file: {filepath}")
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Assuming each JSON file contains a list of documents
                if isinstance(data, list):
                    combined_dataset.extend(data)
                else:
                    # If a file contains a single object, wrap it in a list
                    combined_dataset.append(data)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from {filename}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while reading {filename}: {e}")

# Define the output filename for the combined dataset
combined_output_filename = os.path.join(output_folder, "combined_dataset.json")

# Save the combined dataset to a new JSON file
with open(combined_output_filename, 'w', encoding='utf-8') as f:
    json.dump(combined_dataset, f, ensure_ascii=False, indent=4)

print(f"Successfully merged {len(combined_dataset)} documents into '{combined_output_filename}'")

# Display the first few entries of the combined dataset as an example
if combined_dataset:
    print(f"\nExample of combined dataset (first {min(3, len(combined_dataset))} documents):")
    display(combined_dataset[:min(3, len(combined_dataset))])
else:
    print("No documents were found or combined.")

Reading file: collected_json_data/c4_en_documents.json
Reading file: collected_json_data/fineweb_documents.json
Reading file: collected_json_data/combined_dataset.json
Reading file: collected_json_data/redpajama_v2_sample_documents.json
Reading file: collected_json_data/finepdfs_documents.json
Successfully merged 224 documents into 'collected_json_data/combined_dataset.json'

Example of combined dataset (first 3 documents):


[{'text': 'I thought I was going to finish the 3rd season of the Wire tonight.\nBut there was a commentary on episode 11, so I had to re-watch Middle Ground with the commentary. Hopefully I can finish the season next weekend.',
  'metadata': {'url': 'https://karaokegal.livejournal.com/1773485.html',
   'timestamp': '2019-04-18 14:16:05',
   'c4_id': None}},
 {'text': 'Biomedics 1 Day Extra are daily replacement disposable contact lenses by CooperVision Hydron. Buy one box of 90 lenses.\nBiomedics 1 Day Extra contacts give you all the convenience of a daily disposable lens with no need for solutions, cases or cleaning and are perfect for the occasional wear. These lenses have greater comfort handling with superior ease of insertion and removal.\nBiomedic 1 Day Extra are also marketed under various other brand names including Clear Choice 1-day, Ascend 1-day, easyvision CLARISION SPHERE, Clearsight 1 Day and ProView Daily Disposable.',
  'metadata': {'url': 'https://www.webcontacts.com.a

In [1]:
print("""# Dataset Extraction Process Overview\n\nThis notebook extracts data from four different Hugging Face datasets: `fineweb`, `finepdfs`, `RedPajama-Data-V2 sample`, and `C4 (Colossal, Cleaned Common Crawl)` by applying a unified collection pipeline. Each dataset was processed to obtain a specified number of samples (50 for FineWeb and FinePDFs, 100 for RedPajama-V2, and 30 for C4), which were then saved into individual JSON files and finally merged into a single combined dataset.\n\n## Key Steps and Libraries Used:\n\n1.  **Libraries**: The extraction heavily relied on `requests` for HTTP requests, `gzip` and `json` for handling compressed JSON data, `langdetect` for language identification, and `datasets` and `datatrove` for accessing and streaming data from Hugging Face.\n\n2.  **URL Validation (`check_url_status`)**: A custom Python function `check_url_status` was defined at the beginning of the notebook. This function uses a `HEAD` request to verify if a given URL is accessible and returns a status code other than 404, ensuring that collected links are still active.\n\n3.  **Dataset-Specific Extraction and Filtering**:\n    *   **FineWeb**: This dataset was processed using `datatrove.pipeline.readers.ParquetReader`. Documents were streamed, and each was filtered based on the presence of a URL and text content, a valid URL status (using `check_url_status`), and English language detection (using `langdetect`). 50 random samples were then selected and saved to `fineweb_documents.json`.\n    *   **FinePDFs**: This dataset was loaded directly using `datasets.load_dataset` in streaming mode, specifically targeting the `eng_Latn` configuration. Similar to FineWeb, documents were filtered for URL and content existence, URL validity, and English language. The first 50 documents that met these criteria were collected and saved to `finepdfs_documents.json`.\n    *   **RedPajama-Data-V2 Sample**: Due to specific formatting, this dataset's `.jsonl.gz` files were directly downloaded and processed from the Hugging Face Hub using `requests` and `gzip`. Documents were extracted from `en_head.json.gz` and `en_middle.json.gz` for multiple shards. Filtering involved checking for URL and `raw_content` presence, and valid URL status. A total of 100 random samples were collected from the valid documents across all processed shards and saved to `redpajama_v2_sample_documents.json`.\n    *   **C4 (Colossal, Cleaned Common Crawl)**: This dataset was loaded using `datasets.load_dataset` in streaming mode for the 'en' configuration. Documents were filtered for URL and text content, URL validity, and English language. The first 30 documents passing these filters were collected and saved to `c4_en_documents.json`.\n\n4.  **Language Detection**: For all datasets that contained a 'text' field, `langdetect.detect` was used to ensure that only English content was included in the samples. A minimum text length was also enforced for more reliable detection.\n\n5.  **Data Organization**: After individual files were created, a new folder named `collected_json_data` was created. All the `_documents.json` files from the individual dataset extractions were then moved into this folder.\n\n6.  **Unified Dataset Creation**: Finally, all JSON files within the `collected_json_data` folder were read, and their contents (lists of documents) were combined into a single Python list. This consolidated data was then saved as `combined_dataset.json` within the same folder, providing a single, unified dataset for further analysis or use.""")

# Dataset Extraction Process Overview

This notebook extracts data from four different Hugging Face datasets: `fineweb`, `finepdfs`, `RedPajama-Data-V2 sample`, and `C4 (Colossal, Cleaned Common Crawl)` by applying a unified collection pipeline. Each dataset was processed to obtain a specified number of samples (50 for FineWeb and FinePDFs, 100 for RedPajama-V2, and 30 for C4), which were then saved into individual JSON files and finally merged into a single combined dataset.

## Key Steps and Libraries Used:

1.  **Libraries**: The extraction heavily relied on `requests` for HTTP requests, `gzip` and `json` for handling compressed JSON data, `langdetect` for language identification, and `datasets` and `datatrove` for accessing and streaming data from Hugging Face.

2.  **URL Validation (`check_url_status`)**: A custom Python function `check_url_status` was defined at the beginning of the notebook. This function uses a `HEAD` request to verify if a given URL is accessible and returns a